In [1]:
# CELL 1 — Load, join, reconcile
import pandas as pd, numpy as np
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint

WORKBOOK = "FinTrust_Week2_Data_Quality_Workbook.xlsx"
customer = pd.read_excel(WORKBOOK, sheet_name="Customer_Cleaned")
transaction = pd.read_excel(WORKBOOK, sheet_name="Transaction_Cleaned")
transaction["Transaction_DateTime"] = pd.to_datetime(transaction["Transaction_DateTime"])
df = transaction.merge(customer, on="Customer_ID", how="left", validate="many_to_one")

df["success"] = (df.Transaction_Status == "Successful").astype(int)
df["risk"]    = (df.Risk_Review_Flag == "Yes").astype(int)
df["intl"]    = (df.International_Transaction == "Yes").astype(int)

assert len(df) == 12000 and df.Customer_ID.nunique() == 1500
assert np.isclose(df.Amount_NGN.sum(), 560477354.85, atol=0.01)
print("Reconciliation passed")

Reconciliation passed


In [2]:
# CELL 2 — KPI table (expected values in comments)
kpis = {
 "Total Customers": customer.Customer_ID.nunique(),                 # 1,500
 "Total Transactions": len(df),                                     # 12,000
 "Total Value (NGN)": round(df.Amount_NGN.sum(), 2),                # 560,477,354.85
 "Average Value (NGN)": round(df.Amount_NGN.mean(), 2),             # 46,706.45
 "Median Value (NGN)": round(df.Amount_NGN.median(), 2),            # 10,305.94
 "Success Rate %": round(df.success.mean()*100, 2),                 # 90.47
 "Risk Review Rate %": round(df.risk.mean()*100, 2),                # 19.60
 "International %": round(df.intl.mean()*100, 2),                   # 4.00
}
pd.Series(kpis)

,0
Total Customers,1.500000e+03
Total Transactions,1.200000e+04
Total Value (NGN),5.604774e+08
Average Value (NGN),4.670645e+04
Median Value (NGN),1.030594e+04
Success Rate %,9.047000e+01
Risk Review Rate %,1.960000e+01
International %,4.000000e+00


In [3]:
# CELL 3 — Trend per active day (replaces the Week 3 month-over-month view)
df["Month"] = df.Transaction_DateTime.dt.to_period("M")
m = df.groupby("Month").agg(txns=("Transaction_ID","count"), value=("Amount_NGN","sum"),
        days=("Transaction_DateTime", lambda s: s.dt.normalize().nunique()))
m["txns_per_day"] = m.txns / m.days
m["value_per_day"] = m.value / m.days
m["value_per_day_change_%"] = m.value_per_day.pct_change() * 100   # +3.25, +0.81
m.round(2)

,txns,value,days,txns_per_day,value_per_day,value_per_day_change_%
Month,,,,,,
2026-01,4133,1.884894e+08,31,133.32,6080304.34,NaN
2026-02,3734,1.757869e+08,28,133.36,6278103.46,3.25
2026-03,4133,1.962010e+08,31,133.32,6329065.26,0.81


In [4]:
# CELL 4 — Risk drivers: decile profile + combined model
df["decile"] = pd.qcut(df.Amount_NGN, 10, labels=False) + 1
print(df.groupby("decile").risk.mean().mul(100).round(1))          # deciles 1-9: 16-19, decile 10: 36.0

p90 = df.Amount_NGN.quantile(0.9)                                   # 143,012.07
df["top10"] = (df.Amount_NGN > p90).astype(int)

model = smf.logit("risk ~ C(Transaction_Type, Treatment('Card Purchase')) "
                  "+ C(Channel, Treatment('Mobile App')) + intl + top10", data=df).fit(disp=0)
out = pd.DataFrame({"OR": np.exp(model.params),
                    "CI_low": np.exp(model.conf_int()[0]),
                    "CI_high": np.exp(model.conf_int()[1]),
                    "p": model.pvalues}).round(3)
out    # Transfer 2.40, Cash Withdrawal 2.20, intl 2.72, top10 2.22; channels not significant at 5%

decile
1     16.1
2     19.6
3     17.5
4     17.7
5     17.2
6     17.8
7     17.9
8     18.8
9     17.5
10    36.0
Name: risk, dtype: float64


,OR,CI_low,CI_high,p
Intercept,0.132,0.117,0.149,0.000
"C(Transaction_Type, Treatment('Card Purchase'))[T.Airtime/Data]",1.253,1.034,1.518,0.021
"C(Transaction_Type, Treatment('Card Purchase'))[T.Bill Payment]",1.032,0.856,1.244,0.741
"C(Transaction_Type, Treatment('Card Purchase'))[T.Cash Withdrawal]",2.195,1.868,2.579,0.000
"C(Transaction_Type, Treatment('Card Purchase'))[T.Deposit]",1.083,0.900,1.305,0.399
"C(Transaction_Type, Treatment('Card Purchase'))[T.Transfer]",2.398,2.103,2.734,0.000
"C(Channel, Treatment('Mobile App'))[T.ATM]",1.141,0.994,1.310,0.061
"C(Channel, Treatment('Mobile App'))[T.POS]",0.954,0.840,1.084,0.470
"C(Channel, Treatment('Mobile App'))[T.USSD]",0.870,0.718,1.054,0.155
"C(Channel, Treatment('Mobile App'))[T.Web]",1.141,0.997,1.304,0.054


In [5]:
# CELL 5 — Mobile App: three views of the same question
mob = df.Channel == "Mobile App"
chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(mob, df.success), correction=False)
print(f"Mobile vs rest: chi2={chi2:.3f}, p={p:.4f}  (Bonferroni x5 = {min(1, p*5):.3f})")   # p=0.0213 -> 0.107
print("All 5 channels omnibus p = %.4f" % stats.chi2_contingency(pd.crosstab(df.Channel, df.success))[1])  # 0.1428

for name, g in [("Mobile App", df[mob]), ("Other", df[~mob])]:
    lo, hi = proportion_confint(g.success.sum(), len(g), method="wilson")
    print(f"{name}: {g.success.mean()*100:.2f}%  CI [{lo*100:.2f}, {hi*100:.2f}]")

gap = df.groupby(["Month", mob]).success.mean().unstack().mul(100)
gap["gap_pts"] = gap[False] - gap[True]
gap.round(2)       # gap shrinks 2.30 -> 1.27 -> 0.15

Mobile vs rest: chi2=5.299, p=0.0213  (Bonferroni x5 = 0.107)
All 5 channels omnibus p = 0.1428
Mobile App: 89.75%  CI [88.89, 90.55]
Other: 91.00%  CI [90.30, 91.65]


Channel,False,True,gap_pts
Month,,,
2026-01,91.00,88.70,2.30
2026-02,91.43,90.16,1.27
2026-03,90.60,90.45,0.15


In [6]:
# CELL 6 — Segments, weekday, status: tests for "no difference"
groups = [g.Amount_NGN.values for _, g in df.groupby("Customer_Segment")]
print("Segments  Kruskal p=%.4f | ANOVA p=%.4f" % (stats.kruskal(*groups)[1], stats.f_oneway(*groups)[1]))  # 0.196 | 0.608
prem = df[df.Customer_Segment=="Premium"].Amount_NGN; eve = df[df.Customer_Segment=="Everyday"].Amount_NGN
print("Premium vs Everyday  Welch p=%.4f | Mann-Whitney p=%.4f" % (stats.ttest_ind(prem, eve, equal_var=False)[1],
                                                                  stats.mannwhitneyu(prem, eve)[1]))   # 0.743 | 0.672
df["Weekday"] = df.Transaction_DateTime.dt.day_name()
print("Weekday amounts Kruskal p=%.4f" % stats.kruskal(*[g.Amount_NGN for _, g in df.groupby("Weekday")])[1])  # 0.947
print("Status vs amount Kruskal p=%.4f" % stats.kruskal(*[g.Amount_NGN for _, g in df.groupby("Transaction_Status")])[1])  # 0.265
print("Type vs success chi-square p=%.4f" % stats.chi2_contingency(pd.crosstab(df.Transaction_Type, df.success))[1])      # 0.828

Segments  Kruskal p=0.1961 | ANOVA p=0.6081
Premium vs Everyday  Welch p=0.7429 | Mann-Whitney p=0.6722
Weekday amounts Kruskal p=0.9470
Status vs amount Kruskal p=0.2646
Type vs success chi-square p=0.8283


In [7]:
# CELL 7 — Is the "top quartile = 51% of value" pattern a customer behaviour or just skew?
per_cust = df.groupby("Customer_ID").Amount_NGN.sum().sort_values(ascending=False)
q = len(per_cust) // 4
observed = per_cust.iloc[:q].sum() / per_cust.sum()                 # 51.06%

rng = np.random.default_rng(42)
amounts = df.Amount_NGN.values
null = []
for _ in range(500):
    shuffled = rng.permutation(df.Customer_ID.values)               # reassign transactions to customers at random
    s = pd.Series(amounts).groupby(shuffled).sum().sort_values(ascending=False)
    null.append(s.iloc[:q].sum() / s.sum())
print(f"Observed {observed:.2%} | random baseline mean {np.mean(null):.2%} | 95% range "
      f"[{np.percentile(null,2.5):.2%}, {np.percentile(null,97.5):.2%}]")   # 51.06% | 51.03% | [50.01%, 52.03%]

Observed 51.06% | random baseline mean 51.03% | 95% range [50.01%, 52.03%]
